# Laboratorio: mínimos cuadrados y ajuste de datos

En este laboratorio conectaremos la geometría de la proyección con el ajuste numérico. Al finalizar podrás:

- explorar cómo cambia la suma de cuadrados al mover una recta;
- construir una matriz de diseño y calcular un ajuste;
- verificar que el residuo es ortogonal al espacio de columnas;
- comparar modelos mediante SSE y $R^2$;
- reconocer qué ocurre cuando los coeficientes no son únicos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interact

np.set_printoptions(precision=6, suppress=True)

## 1. Explorar la función objetivo

Para una recta $\widehat y=at+c$, cada residuo es $r_i=y_i-\widehat y_i$. Los segmentos verticales de la gráfica representan esos residuos; la función objetivo es $\mathrm{SSE}=\sum_i r_i^2$. Mueve los controles y busca visualmente una recta con SSE pequeña.

In [ ]:
t = np.array([0.5, 1.2, 2.0, 3.0, 4.0])
y = np.array([1.2, 1.9, 3.0, 2.4, 3.2])

def explorar_recta(a=0.5, c=1.0):
    y_hat = a * t + c
    residuos = y - y_hat
    sse = residuos @ residuos

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(t, y, color="#14578c", zorder=3, label="datos")
    ax.plot(t, y_hat, color="#be2d37", label=fr"$\widehat y={a:.2f}t+{c:.2f}$")
    ax.vlines(t, y_hat, y, colors="gray", linestyles="--", alpha=0.8)
    ax.set_title(fr"Suma de cuadrados de residuos: ${sse:.4f}$")
    ax.set_xlabel("t")
    ax.set_ylabel("y")
    ax.grid(alpha=0.2)
    ax.legend()
    plt.show()

interact(
    explorar_recta,
    a=FloatSlider(min=-1, max=2, step=0.05, value=0.5, description="a"),
    c=FloatSlider(min=-1, max=4, step=0.05, value=1.0, description="c"),
)

## 2. Matriz de diseño y solución

Para la base $(t,1)$, la matriz de diseño es $X=[\,t\;\mathbf 1\,]$. Usaremos `numpy.linalg.lstsq`, que calcula mínimos cuadrados sin formar explícitamente la inversa de $X^TX$.

In [ ]:
X = np.column_stack([t, np.ones_like(t)])
beta, suma_residuos, rango, valores_singulares = np.linalg.lstsq(X, y, rcond=None)
a, c = beta

print("Matriz de diseño X:\n", X)
print(f"\nRecta ajustada: y = {a:.6f} t + {c:.6f}")
print("Rango de X:", rango)
print("Valores singulares informados por lstsq:", valores_singulares)

### Verificación geométrica

El vector ajustado es $\widehat y=X\beta$ y el residuo es $r=y-\widehat y$. Las ecuaciones normales equivalen a $X^Tr=0$.

In [ ]:
y_hat = X @ beta
r = y - y_hat

print("Valores ajustados:", y_hat)
print("Residuos:", r)
print("X.T @ r:", X.T @ r)

assert np.allclose(X.T @ r, 0)

In [ ]:
# La solución de las ecuaciones normales coincide en este ejemplo bien condicionado.
beta_normales = np.linalg.solve(X.T @ X, X.T @ y)
print("lstsq:             ", beta)
print("ecuaciones normales:", beta_normales)
print("diferencia:         ", beta - beta_normales)

assert np.allclose(beta, beta_normales)

In [ ]:
explorar_recta(a, c)

## 3. Un ejemplo de la hoja teórica

Para $(1,2)$, $(2,2)$ y $(3,4)$, la recta exacta de mínimos cuadrados es $y=t+\frac23$. Comprobaremos también que el residuo es ortogonal a las columnas $(1,2,3)^T$ y $(1,1,1)^T$.

In [ ]:
t_ej = np.array([1.0, 2.0, 3.0])
y_ej = np.array([2.0, 2.0, 4.0])
X_ej = np.column_stack([t_ej, np.ones_like(t_ej)])
beta_ej = np.linalg.solve(X_ej.T @ X_ej, X_ej.T @ y_ej)
r_ej = y_ej - X_ej @ beta_ej

print("X.T @ X =\n", X_ej.T @ X_ej)
print("X.T @ y =", X_ej.T @ y_ej)
print("(a, c) =", beta_ej)
print("residuo =", r_ej)
print("X.T @ residuo =", X_ej.T @ r_ej)

assert np.allclose(beta_ej, [1, 2/3])
assert np.allclose(X_ej.T @ r_ej, 0)

## 4. Funciones base generales

La siguiente función construye $X_{ij}=\varphi_j(t_i)$. Esto permite cambiar el modelo sin cambiar el algoritmo de mínimos cuadrados.

In [ ]:
def matriz_diseno(puntos, funciones):
    puntos = np.asarray(puntos, dtype=float)
    columnas = []
    for phi in funciones:
        columna = np.asarray(phi(puntos), dtype=float)
        if columna.ndim == 0:
            columna = np.full(puntos.shape, columna, dtype=float)
        columnas.append(columna)
    return np.column_stack(columnas)

base_lineal = [lambda z: np.ones_like(z), lambda z: z]
base_cuadratica = [lambda z: np.ones_like(z), lambda z: z, lambda z: z**2]

In [ ]:
X_lineal = matriz_diseno(t, base_lineal)
X_cuadratica = matriz_diseno(t, base_cuadratica)

beta_lineal = np.linalg.lstsq(X_lineal, y, rcond=None)[0]
beta_cuadratica = np.linalg.lstsq(X_cuadratica, y, rcond=None)[0]

print("Coeficientes [1, t]:", beta_lineal)
print("Coeficientes [1, t, t^2]:", beta_cuadratica)

In [ ]:
t_plot = np.linspace(t.min(), t.max(), 250)
y_lineal_plot = matriz_diseno(t_plot, base_lineal) @ beta_lineal
y_cuadratica_plot = matriz_diseno(t_plot, base_cuadratica) @ beta_cuadratica

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(t, y, color="#17202a", label="datos")
ax.plot(t_plot, y_lineal_plot, color="#14578c", label="ajuste lineal")
ax.plot(t_plot, y_cuadratica_plot, color="#be2d37", label="ajuste cuadrático")
ax.set_xlabel("t")
ax.set_ylabel("y")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

### 4.1. Ejemplo completo: $f(t)=a+bt+ct^2$

Usaremos los datos $(-2,5)$, $(-1,2)$, $(0,1)$, $(1,2)$ y $(2,4)$. Las columnas de la matriz de diseño corresponden, en ese orden, a $1$, $t$ y $t^2$.

In [ ]:
t_q = np.array([-2., -1., 0., 1., 2.])
z_q = np.array([5., 2., 1., 2., 4.])
X_q = np.column_stack([np.ones_like(t_q), t_q, t_q**2])
beta_q = np.linalg.lstsq(X_q, z_q, rcond=None)[0]
r_q = z_q - X_q @ beta_q

print("X.T @ X =\n", X_q.T @ X_q)
print("X.T @ z =", X_q.T @ z_q)
print("(a, b, c) =", beta_q)
print("residuo =", r_q)
print("X.T @ residuo =", X_q.T @ r_q)

assert np.allclose(beta_q, [38/35, -1/5, 6/7])
assert np.allclose(r_q, np.array([3, -5, -3, 9, -4]) / 35)
assert np.allclose(X_q.T @ r_q, 0)

In [ ]:
t_q_plot = np.linspace(-2.2, 2.2, 250)
z_q_plot = beta_q[0] + beta_q[1] * t_q_plot + beta_q[2] * t_q_plot**2

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(t_q, z_q, color="#14578c", label="datos", zorder=3)
ax.plot(t_q_plot, z_q_plot, color="#be2d37", label="ajuste cuadrático")
ax.vlines(t_q, X_q @ beta_q, z_q, colors="gray", linestyles="--")
ax.set_xlabel("t")
ax.set_ylabel("z")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

### 4.2. Ejemplo completo: $f(x,y)=a+bx+cy$

Ajustaremos un plano afín a cuatro observaciones $(x_i,y_i,z_i)$. Cada fila de la matriz de diseño es $[1,x_i,y_i]$.

In [ ]:
puntos_2d = np.array([[0., 0.], [1., 0.], [0., 1.], [1., 1.]])
z_2d = np.array([1., 3., 4., 5.])
X_2d = np.column_stack([np.ones(len(puntos_2d)), puntos_2d])
beta_2d = np.linalg.lstsq(X_2d, z_2d, rcond=None)[0]
r_2d = z_2d - X_2d @ beta_2d

print("X.T @ X =\n", X_2d.T @ X_2d)
print("X.T @ z =", X_2d.T @ z_2d)
print("(a, b, c) =", beta_2d)
print("residuo =", r_2d)
print("X.T @ residuo =", X_2d.T @ r_2d)

assert np.allclose(beta_2d, [5/4, 3/2, 5/2])
assert np.allclose(r_2d, np.array([-1, 1, 1, -1]) / 4)
assert np.allclose(X_2d.T @ r_2d, 0)

In [ ]:
xg, yg = np.meshgrid(np.linspace(-0.1, 1.1, 25), np.linspace(-0.1, 1.1, 25))
zg = beta_2d[0] + beta_2d[1] * xg + beta_2d[2] * yg

fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(puntos_2d[:, 0], puntos_2d[:, 1], z_2d, color="#be2d37", s=45, label="datos")
ax.plot_surface(xg, yg, zg, color="#75aadb", alpha=0.5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Ajuste afín en dos variables")
plt.show()

### 4.3. Solo modelamiento: grado dos en dos variables

Para el modelo completo

$$f(x,y)=a+bx+cy+dx^2+exy+fy^2,$$

cada observación produce la fila

$$[1,\ x_i,\ y_i,\ x_i^2,\ x_i y_i,\ y_i^2].$$

Al apilar las filas se obtiene $X$, se define $\beta=(a,b,c,d,e,f)^T$ y se plantean las ecuaciones normales

$$X^TX\beta=X^Tz.$$

Variantes: sin interacción se elimina la columna $xy$; para $a+bx+cy+dxy$ se usan las columnas $1,x,y,xy$; para $a+bx+cy+dx^2$ se usan $1,x,y,x^2$.

## 5. SSE y coeficiente de determinación

Cuando el modelo contiene una constante y los datos no son todos iguales,

$$R^2=1-\frac{\|y-\widehat y\|^2}{\|y-\overline y\|^2}.$$

In [ ]:
def metricas_ajuste(y_observado, y_ajustado):
    residuo = y_observado - y_ajustado
    sse = residuo @ residuo
    centrado = y_observado - np.mean(y_observado)
    sst = centrado @ centrado
    r2 = 1 - sse / sst
    return sse, r2

y_lineal = X_lineal @ beta_lineal
y_cuadratica = X_cuadratica @ beta_cuadratica

for nombre, ajuste in [("lineal", y_lineal), ("cuadrático", y_cuadratica)]:
    sse, r2 = metricas_ajuste(y, ajuste)
    print(f"{nombre:10s}: SSE = {sse:.6f}, R^2 = {r2:.6f}")

In [ ]:
# Verificación de la descomposición ortogonal SST = SSE + SSR.
y_barra = np.full_like(y, np.mean(y))
sse = np.linalg.norm(y - y_lineal)**2
ssr = np.linalg.norm(y_lineal - y_barra)**2
sst = np.linalg.norm(y - y_barra)**2

print("SST:", sst)
print("SSE + SSR:", sse + ssr)
assert np.allclose(sst, sse + ssr)

## 6. Ajuste único, coeficientes no únicos

Duplicaremos una columna. La matriz pierde rango: habrá distintos vectores de coeficientes que producen exactamente el mismo vector ajustado. `lstsq` devuelve, entre ellos, el de norma mínima.

In [ ]:
A_dep = np.column_stack([t, t, np.ones_like(t)])
beta_min, _, rango_dep, _ = np.linalg.lstsq(A_dep, y, rcond=None)
direccion_nula = np.array([1.0, -1.0, 0.0])
beta_alternativa = beta_min + 3 * direccion_nula

print("Rango:", rango_dep, "de", A_dep.shape[1], "columnas")
print("Coeficientes de norma mínima:", beta_min)
print("Otros coeficientes:          ", beta_alternativa)
print("Normas:", np.linalg.norm(beta_min), np.linalg.norm(beta_alternativa))
print("Diferencia entre ajustes:", np.linalg.norm(A_dep @ beta_min - A_dep @ beta_alternativa))

assert np.allclose(A_dep @ direccion_nula, 0)
assert np.allclose(A_dep @ beta_min, A_dep @ beta_alternativa)

## 7. Extensión: regularización cuadrática

Para $\alpha>0$, el sistema $(A^TA+\alpha I)\beta=A^Ty$ tiene solución única. Observa cómo cambia el tamaño de los coeficientes al variar $\alpha$.

In [ ]:
def ridge(A, b, alpha):
    n = A.shape[1]
    return np.linalg.solve(A.T @ A + alpha * np.eye(n), A.T @ b)

for alpha in [1e-3, 0.1, 1.0, 10.0]:
    beta_ridge = ridge(A_dep, y, alpha)
    sse = np.linalg.norm(y - A_dep @ beta_ridge)**2
    print(f"alpha={alpha:>6g}: beta={beta_ridge}, ||beta||={np.linalg.norm(beta_ridge):.5f}, SSE={sse:.5f}")

## 8. Actividades

1. Usa el control interactivo y compara tu mejor estimación visual con `beta`.
2. Sustituye los datos por $(0,1)$, $(1,2)$ y $(2,2)$. Calcula la recta y verifica $X^Tr=0$.
3. Agrega la función base $t^3$ y compara SSE y $R^2$. ¿Un valor mayor de $R^2$ garantiza una mejor explicación fuera de los datos observados?
4. Construye el modelo $\beta_0+\beta_1\sin t+\beta_2\cos t$ para datos periódicos.
5. Cambia la columna duplicada por una columna casi duplicada y compara las soluciones obtenidas con ecuaciones normales y con `lstsq`.
6. Explica geométricamente por qué todos los vectores `beta_min + s * direccion_nula` producen el mismo ajuste.